In [1]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [2]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

In [3]:
import os

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000"))
mlflow.set_experiment("Breast_Cancer_Experiment")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
lr_configs = [
    {"C": 1.0,  "solver": "lbfgs",  "version": "lr-v1"},
    {"C": 0.1,  "solver": "lbfgs",  "version": "lr-v2"},
    {"C": 10.0, "solver": "lbfgs",  "version": "lr-v3"},
]

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [6]:
for cfg in lr_configs:
    with mlflow.start_run(run_name=cfg["version"]):
        mlflow.set_tag("model_type", "LogisticRegression")
        mlflow.set_tag("version", cfg["version"])

        lr = LogisticRegression(max_iter=1000, C=cfg["C"], solver=cfg["solver"])
        lr.fit(X_train, y_train)
        y_pred = lr.predict(X_test)

        mlflow.log_param("model", "LogisticRegression")
        mlflow.log_param("solver", cfg["solver"])
        mlflow.log_param("C", cfg["C"])

        mlflow.log_metric("accuracy",  accuracy_score(y_test, y_pred))
        mlflow.log_metric("precision", precision_score(y_test, y_pred))
        mlflow.log_metric("recall",    recall_score(y_test, y_pred))
        mlflow.log_metric("f1",        f1_score(y_test, y_pred))

        mlflow.sklearn.log_model(lr, "model")

/usr/local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/03/05 17:57:16 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
2026/03/05 17:57:16 INFO mlflow.tracking._tracking_service.client: 🏃 View run lr-v1 at: http://mlflow:5000/#/experiments/119455075363281320/runs/f70349d62b624e0cb05e46418ed3a845.
2026/03/05 17:57:16 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/119455075363281320

In [7]:
rf_configs = [
    {"n_estimators": 50,  "max_depth": None, "version": "rf-v1"},
    {"n_estimators": 100, "max_depth": None, "version": "rf-v2"},
    {"n_estimators": 100, "max_depth": 5,    "version": "rf-v3"},
]

In [8]:
for cfg in rf_configs:
    with mlflow.start_run(run_name=cfg["version"]):
        mlflow.set_tag("model_type", "RandomForest")
        mlflow.set_tag("version", cfg["version"])

        rf = RandomForestClassifier(
            n_estimators=cfg["n_estimators"],
            max_depth=cfg["max_depth"],
            random_state=42,
        )
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)

        mlflow.log_param("model", "RandomForest")
        mlflow.log_param("n_estimators", cfg["n_estimators"])
        mlflow.log_param("max_depth", cfg["max_depth"])

        mlflow.log_metric("accuracy",  accuracy_score(y_test, y_pred))
        mlflow.log_metric("precision", precision_score(y_test, y_pred))
        mlflow.log_metric("recall",    recall_score(y_test, y_pred))
        mlflow.log_metric("f1",        f1_score(y_test, y_pred))

        mlflow.sklearn.log_model(rf, "model")


/usr/local/lib/python3.11/site-packages/_distutils_hack/__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
2026/03/05 17:58:13 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the 

In [9]:
print("done")

done
